In [2]:
from pyspark.sql import SparkSession

In [1]:
from pyspark.sql import SparkContext #stara wersja

ImportError: cannot import name 'SparkContext' from 'pyspark.sql' (/usr/local/spark/python/pyspark/sql/__init__.py)

In [3]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [7]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

kategorie = (
    df.groupBy("category")
    .agg(
        _sum("amount").alias("suma_PLN"),
        _min("amount").alias("min_PLN"),
        _max("amount").alias("max_PLN")
    )
    .orderBy("category")
)
kategorie.show()

+-----------+------------------+-------+-------+
|   category|          suma_PLN|min_PLN|max_PLN|
+-----------+------------------+-------+-------+
|elektronika|1520770.6899999995|    9.0| 9999.0|
|    książki| 851382.0799999991|    5.0|9107.25|
|     odzież| 849877.5500000007|    5.0|9696.63|
|    żywność| 789514.4300000003|    5.0|6916.92|
+-----------+------------------+-------+-------+



In [10]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [11]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [12]:
from pyspark.sql.functions import window, count, round as _round

okna_30m_sklepy = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy("window", "store")
)
okna_30m_sklepy.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [13]:
from pyspark.sql.functions import desc

(
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma_PLN"))
    .orderBy(desc("suma_PLN"))
    .show(1, truncate=False) # .show(1) pokazuje tylko pierwszy, czyli NAJWYŻSZY wynik!
)

+------------------------------------------+---------+
|window                                    |suma_PLN |
+------------------------------------------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86|
+------------------------------------------+---------+
only showing top 1 row



In [14]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [15]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# W oknach sliding przedziału czasu zachodzą na siebie, co sprawia, że dana trasakcja przypisana jest do więcej niż jednego okna równocześnie.

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: 
# groupBy("store") agreguje całą historię transakcji do jednego wyniku dla danego sklepu (na przestrzeni wszystkich dostępnych godzin).
# groupBy(window(...), "store") najpierw tnie całą historię na podane kawałki czasu (np. na godziny), a dopiero w ich wnętrzu podsumowuje dane dla poszczególnych sklepów.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: Transakcja wykonana o 09:30 znajdzie się w 2 oknach. 
# Spark operuje na przedziałach domknięto-otwartych [start, koniec). 
# Transakcja z 09:30 wpadnie więc w okno [09:00 - 10:00) oraz w okno [09:30 - 10:30).

In [16]:
# Praca domowa 1
# Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

from pyspark.sql.functions import avg

(
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(avg("amount").alias("srednia_PLN"))
    .orderBy("srednia_PLN") # sortowanie domyślnie rosnące, więc najniższa jest na górze
    .show(1, truncate=False)
)


+------------------------------------------+-----------------+
|window                                    |srednia_PLN      |
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.0118407310706|
+------------------------------------------+-----------------+
only showing top 1 row



In [17]:
# Praca domowa 2
# Policz ile transakcji per kategoria było w oknie 09:00–09:30.

(
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
    .filter(col("window.start").like("%09:00:00%")) # wyciągamy tylko konkretne okno
    .show(truncate=False)
)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



In [18]:
# Praca domowa 3
# Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

(
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
    .show(1, truncate=False)
)

+------------------------------------------+---------+
|window                                    |liczba_tx|
+------------------------------------------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |
+------------------------------------------+---------+
only showing top 1 row

